# 02-1b. 딥러닝·통계 모델 예측 (같은 5주 시험)

02-2 노트북의 후보(기준 곡선, LightGBM, 선형 회귀, 칼만 필터, Chronos-2)에 더해, **부록 파이프라인에서 쓴 모델 계열**을 15분 분석의 조건(7/1 이후 자료, 15분 단위, 같은 5주 시간순 평가)에서 다시 예측합니다. 비교와 검정은 02-2 노트북 3-6절에서 합니다.

| 계열 | 모델 | 쉬운 설명 |
|---|---|---|
| 통계 | **MSTL** | 전력을 하루 주기·일주일 주기·추세로 나눠 각각 이어 그림 (하루 앞만, 아래 설명) |
| 통계 | **ARIMA (+달력·가동 계획)** | 최근 값의 흐름을 따라가는 시계열 모델에 시각(푸리에 항)·하루 종류를 더함. 부록 파이프라인과 같은 구성 |
| 통계 | **ARIMA (+기준 곡선)** | 02-2의 기준 곡선을 입력으로 넣고, 곡선과의 차이를 ARIMA로 이어 감 |
| 딥러닝 | **NHITS** | 과거 7일을 보고 여러 시간 규모로 나눠 예측하는 신경망 |
| 딥러닝 | **TFT** | 어떤 입력·시점에 주목할지 스스로 정하는 어텐션 기반 신경망 |

LSTM(순환 신경망)도 시도했지만 이 CPU 환경에서 시험 한 주를 학습하는 데 40분 넘게 걸려 15분 비교에서는 뺐습니다. LSTM 결과는 부록 파이프라인(1시간 최대 예측)에 있습니다.

**공통 조건**
- 각 시험 주는 **그 주 이전 자료로만** 학습합니다(02-2와 같은 확장 창). 학습 기간의 빈 값은 그 주 이전 자료로 만든 기준 곡선으로 채우고, 신경망은 그 칸을 학습 손실에서 뺍니다.
- 미래에 아는 정보는 Chronos-2 (+가동 계획)과 같습니다: 시각·요일(사인·코사인), 하루 종류(토요일·휴일), 전날 가동.
- 하루 앞은 매일 자정에 96칸, 1시간 앞은 매 정시에 4칸을 예측합니다. 예측 시점까지의 실제 전력만 씁니다.
- 신경망 설정은 부록 파이프라인을 따릅니다: 학습률 1e-3, 마지막 7일을 검증으로 50단계마다 확인해 3번 연속 나아지지 않으면 멈춤(조기 종료). NHITS는 과거 7일·최대 400단계, TFT는 CPU 시간 때문에 과거 2일·최대 150단계, 목표값은 학습 기간 평균·표준편차로 표준화, 시드 42.

> **실행 환경**: `requirements.txt` 환경. 모델·예측 범위마다 결과를 `outputs/predictions/extra_parts/`에 저장하고, 저장된 결과가 있으면 다시 계산하지 않고 불러옵니다. 모두 새로 계산하면 CPU로 수 시간 걸립니다(아래 표의 계산 시간).

## 0. 준비

In [1]:
import logging, os, time, warnings
warnings.filterwarnings("ignore")
for _lg in ("pytorch_lightning", "lightning", "lightning.pytorch", "lightning.fabric", "neuralforecast"):
    logging.getLogger(_lg).setLevel(logging.ERROR)
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS, TFT
from statsforecast import StatsForecast
from statsforecast.models import MSTL, AutoETS, AutoARIMA

SEED = 42
torch.set_num_threads(int(os.environ.get("N_THREADS", "4")))
PRED_DIR = Path("../outputs/predictions")
PART_DIR = PRED_DIR / "extra_parts"; PART_DIR.mkdir(parents=True, exist_ok=True)
slots = pd.read_csv("../data/processed/slots_15min.csv", encoding="utf-8-sig", parse_dates=["ts", "date"], index_col="ts")
slots["칸번호"] = (slots["시각"] * 4).astype(int)
slots["요일"] = slots.index.dayofweek
slots["하루종류코드"] = slots["하루종류"].map({"평일가동": 0, "토요일": 1, "휴일": 2})
slots["전날가동"] = slots["전날가동"].astype(int)
TEST_STARTS = pd.to_datetime(["2021-08-09", "2021-08-16", "2021-08-23", "2021-08-30", "2021-09-06"])
TEST_ENDS = list(TEST_STARTS[1:]) + [pd.Timestamp("2021-09-15")]
CURVE_KEYS = ["하루종류코드", "전날가동", "칸번호"]

# 미래에 미리 아는 정보 (Chronos-2 (+가동 계획)과 같은 정보)
ang_d = 2 * np.pi * slots["칸번호"] / 96
ang_w = 2 * np.pi * slots["요일"] / 7
FUT = pd.DataFrame({"시각_sin": np.sin(ang_d), "시각_cos": np.cos(ang_d), "요일_sin": np.sin(ang_w), "요일_cos": np.cos(ang_w),
                    "토요일": (slots["하루종류코드"] == 1).astype(float), "휴일": (slots["하루종류코드"] == 2).astype(float),
                    "전날가동": slots["전날가동"].astype(float)}, index=slots.index)
FUT_COLS = list(FUT.columns)
# ARIMA (+달력·가동 계획): 하루 주기 푸리에 항(1~4차)을 평일 가동일과 그 밖의 날로 나눠 넣음
op = (slots["하루종류코드"] == 0).astype(float)
ARX = {"토요일": FUT["토요일"], "휴일": FUT["휴일"], "전날가동": FUT["전날가동"]}
for k in range(1, 5):
    for nm, g in [("가동", op), ("비가동", 1 - op)]:
        ARX[f"sin{k}_{nm}"] = np.sin(k * ang_d) * g
        ARX[f"cos{k}_{nm}"] = np.cos(k * ang_d) * g
ARX = pd.DataFrame(ARX, index=slots.index)
print(slots.shape, "| 미래에 아는 정보:", FUT_COLS)

2026-10-05 21:20:34,910	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


2026-10-05 21:20:35,371	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


(7296, 19) | 미래에 아는 정보: ['시각_sin', '시각_cos', '요일_sin', '요일_cos', '토요일', '휴일', '전날가동']


In [2]:
def base_curve(st):
    return slots[(slots.index < st) & slots["정답사용"]].groupby(CURVE_KEYS)["전력"].mean().rename("기준곡선")

def series_until(st, en):
    """en 직전까지의 전력. 빈 값은 st 이전 자료의 기준 곡선으로 채움(시험 주 정보 사용 안 함). 학습 손실에 쓸 칸 표시"""
    s = slots[slots.index < en]
    y = s["전력"].fillna(s.join(base_curve(st), on=CURVE_KEYS)["기준곡선"])
    return y, s["정답사용"].astype(float)

def to_pred(cv, col, cutoff_col="cutoff"):
    return pd.DataFrame({"발행시각": cv[cutoff_col] + pd.Timedelta(minutes=15), "대상시각": cv["ds"], "예측": cv[col].to_numpy()})

def run_stats(which, h, st, en):
    y, _ = series_until(st, en)
    df = pd.DataFrame({"unique_id": "공장", "ds": y.index, "y": y.to_numpy()})
    if which == "MSTL":
        model = MSTL(season_length=[96, 672], trend_forecaster=AutoETS(model="ZZN"), alias=which)
    elif which == "ARIMA (+달력·가동 계획)":
        model = AutoARIMA(season_length=1, alias=which)
        df = pd.concat([df, ARX.loc[y.index].reset_index(drop=True)], axis=1)
    else:   # ARIMA (+기준 곡선)
        model = AutoARIMA(season_length=1, alias=which)
        df["기준곡선"] = slots.loc[y.index].join(base_curve(st), on=CURVE_KEYS)["기준곡선"].to_numpy()
    n = int((y.index >= st).sum())
    cv = StatsForecast(models=[model], freq="15min", n_jobs=1).cross_validation(df=df, h=h, n_windows=n // h, step_size=h, refit=False)
    assert cv["ds"].min() == st and len(cv) == n
    return to_pred(cv, which)

# CPU 시간 때문에 단계 상한과 입력 길이를 줄임. TFT는 어텐션 계산이 무거워 입력 2일, 최대 150단계로 축소
NEURAL = {"NHITS": (NHITS, dict(input_size=672, max_steps=400)),
          "TFT": (TFT, dict(input_size=192, max_steps=150, hidden_size=32))}

def run_neural(which, h, st, en):
    y, mask = series_until(st, en)
    tr = (y.index < st) & (mask > 0)
    mu, sd = y[tr].mean(), y[tr].std()
    frame = pd.DataFrame({"unique_id": "공장", "ds": y.index, "y": ((y - mu) / sd).to_numpy(), "available_mask": mask.to_numpy()})
    frame = pd.concat([frame, FUT.loc[y.index].reset_index(drop=True)], axis=1)
    cls, cfg = NEURAL[which]
    model = cls(h=h, alias=which, futr_exog_list=FUT_COLS, scaler_type="identity", learning_rate=1e-3,
                val_check_steps=50, early_stop_patience_steps=3, random_seed=SEED,
                windows_batch_size=256, inference_windows_batch_size=512, deterministic=True, accelerator="cpu",
                enable_progress_bar=False, logger=False, enable_model_summary=False, **cfg)
    n = int((y.index >= st).sum())
    cv = NeuralForecast(models=[model], freq="15min").cross_validation(df=frame, n_windows=n // h, step_size=h, val_size=672, refit=False)
    assert cv["ds"].min() == st and len(cv) == n
    cv[which] = cv[which] * sd + mu
    return to_pred(cv, which)

KEY = {"MSTL": "mstl", "ARIMA (+달력·가동 계획)": "arima_cal", "ARIMA (+기준 곡선)": "arima_curve", "NHITS": "nhits", "TFT": "tft"}

def run_job(name, h):
    """한 모델·한 예측 범위의 5주 예측. 저장된 결과가 있으면 불러옴"""
    path = PART_DIR / f"{'day' if h == 96 else 'hour'}_{KEY[name]}.csv"
    if path.exists():
        return pd.read_csv(path, encoding="utf-8-sig", parse_dates=["발행시각", "대상시각"]), "저장된 결과"
    fn = run_stats if name in ("MSTL", "ARIMA (+달력·가동 계획)", "ARIMA (+기준 곡선)") else run_neural
    t0 = time.time()
    out = pd.concat([fn(name, h, st, en).assign(모델=name) for st, en in zip(TEST_STARTS, TEST_ENDS)])
    out.to_csv(path, encoding="utf-8-sig", index=False)
    return out, f"{time.time() - t0:.0f}초"

## 1. 하루 앞 예측 (매일 자정, 96칸)

In [3]:
DAY_JOBS = ["MSTL", "ARIMA (+달력·가동 계획)", "ARIMA (+기준 곡선)", "NHITS", "TFT"]
day_out = []
for name in DAY_JOBS:
    df, how = run_job(name, 96)
    day_out.append(df)
    print(f"{name}: {how}")
extra_day = pd.concat(day_out)
assert (extra_day["발행시각"] == extra_day["대상시각"].dt.normalize()).all()       # 자정 발행, 같은 날 96칸
extra_day.drop(columns="발행시각").rename(columns={"대상시각": "ts"}).to_csv(PRED_DIR / "extra_day_ahead.csv", encoding="utf-8-sig", index=False)

MSTL: 저장된 결과
ARIMA (+달력·가동 계획): 저장된 결과
ARIMA (+기준 곡선): 저장된 결과
NHITS: 저장된 결과
TFT: 저장된 결과


## 2. 1시간 앞 예측 (매 정시, 4칸)
MSTL은 매 정시마다 전체 기록을 다시 분해해야 해서 CPU로 1시간 이상 걸려 1시간 앞에서는 뺐습니다.

In [4]:
HOUR_JOBS = ["ARIMA (+달력·가동 계획)", "ARIMA (+기준 곡선)", "NHITS", "TFT"]
hour_out = []
for name in HOUR_JOBS:
    df, how = run_job(name, 4)
    hour_out.append(df)
    print(f"{name}: {how}")
extra_hour = pd.concat(hour_out)
assert (extra_hour["발행시각"].dt.minute == 0).all()
extra_hour.to_csv(PRED_DIR / "extra_hour_ahead.csv", encoding="utf-8-sig", index=False)

# 미리 보기: 평가 대상 칸 MAE (비교·검정은 02-2 노트북)
def preview(p, key):
    d = p.join(slots[["전력", "정답사용"]], on=key)
    d = d[d["정답사용"]]
    return (d["예측"] - d["전력"]).abs().groupby(d["모델"]).mean().round(2)
pd.DataFrame({"하루 앞 MAE": preview(extra_day.rename(columns={"대상시각": "ts"}), "ts"), "1시간 앞 MAE": preview(extra_hour, "대상시각")})

ARIMA (+달력·가동 계획): 저장된 결과
ARIMA (+기준 곡선): 저장된 결과
NHITS: 저장된 결과
TFT: 저장된 결과


,하루 앞 MAE,1시간 앞 MAE
모델,,
ARIMA (+기준 곡선),7.71,6.07
ARIMA (+달력·가동 계획),18.11,13.25
MSTL,11.64,NaN
NHITS,13.96,9.59
TFT,12.18,11.75


## 3. 계산 시간 (참고)
`outputs/predictions/extra_parts/`가 없을 때 5주를 새로 계산한 시간입니다(CPU, 모델별로 따로 실행, 괄호는 사용한 스레드 수).

| 모델 | 하루 앞 | 1시간 앞 |
|---|---|---|
| ARIMA (+기준 곡선) | 2분 (1) | 2분 (1) |
| ARIMA (+달력·가동 계획) | 4분 (1) | 4분 (1) |
| MSTL | 8분 (1) | – |
| NHITS | 17분 (3) | 10분 (3) |
| TFT | 30분 (5) | 20분 (5) |

**해석**
- 모든 후보가 02-2의 최종 모델(하루 앞 7.32kW, 1시간 앞 5.85kW)보다 MAE가 컸습니다. 비교와 검정은 02-2 노트북 3-6절에 있습니다.
- 가장 가까운 후보는 기준 곡선을 입력으로 넣은 ARIMA(7.71 / 6.07kW)였습니다. 가동 계획을 '하루 종류 × 시각' 구조로 담은 기준 곡선이 있어야 시계열 모델도 강해진다는 뜻입니다.
- 과거 기록만 보는 MSTL과, 가동 계획을 받더라도 직전 7일 기록에 크게 기대는 NHITS는 하계 휴무 직후인 1주차에 크게 틀렸습니다(하루 앞 MAE 21·28kW). 직전 7일이 휴무라 입력이 평소와 달랐기 때문입니다.
- **단, NHITS·TFT는 CPU 시간 때문에 줄인 설정의 결과입니다**(NHITS 과거 7일·최대 400단계, TFT 과거 2일·최대 150단계). 1시간 앞에서 두 모델의 오차(9.59·11.75kW)는 직전 전력을 쓰지 않는 기준 곡선만 쓸 때(7.32kW, 02-2 표 11)보다도 커서 학습이 충분하지 않았을 가능성이 큽니다. 이 결과는 줄인 설정에서의 비교이며, 딥러닝 계열 전체에 대한 결론은 아닙니다.